# 01 — Bronze ingestion: eventstream simulator envelopes

| Field | Value |
| ----- | ----- |
| **Sprint** | Sprint 09 v2 — T2.6 |
| **Layer** | `bronze/eventstream/<eventKind>/` |
| **Source** | Fabric Eventstream binding (upstream = Event Hubs `ehihzhhpfsit*`, bound in T2.2). Eventstream materialises the raw envelopes into a lakehouse-managed Delta table (default name `bronze_eventstream_raw`). |
| **Target** | `Tables/bronze/eventstream/<eventKind>/` (Delta, append) — one folder per `eventKind` message property. |
| **Governance** | [ADR-0015](../../../docs/adr/0015-skip-sql-for-mvp-demo.md) (SQL-free MVP path), [ADR-0016](../../../docs/adr/0016-no-phi-in-mvp-demo-scope.md) (no-PHI demo scope, gate applied at silver), [ADR-0013](../../../docs/adr/0013-temporary-us-region-demo-scope.md) (westus2 demo carve-out) |
| **Design spec** | [§4.3 event kinds, §4.6 notebook chain](../../../docs/superpowers/specs/2026-07-02-sprint-09-v2-refinement-design.md) |

## Purpose

Consume the Fabric Eventstream-materialised envelopes and land them **as-is** into
the bronze zone, **routed by the `eventKind` message property** into per-kind
Delta folders. No validation, no schema coercion, no PHI screening at this layer
— those gates live in `02_silver_eventstream.ipynb` (design spec §4.6 table).

## Envelope contract (design spec §4.3)

Every envelope carries:

- `eventKind` (routing key — one of the 7 kinds listed below)
- `eventId`, `hospitalId`, `simulatedAt`, `emittedAt`
- `simRunId`, `seed` (deterministic-replay lineage)
- `payload` (per-eventKind schema, validated at silver)

The full envelope is preserved verbatim in bronze plus a `_lineage_ref` column
carrying `<source>:<ingest_ts>` so silver can trace back to the batch.

## Seven event kinds routed (design spec §4.3)

| `eventKind` | Bronze folder |
| ----------- | ------------- |
| `encounter.admitted` | `bronze/eventstream/encounter.admitted/` |
| `encounter.transitioned` | `bronze/eventstream/encounter.transitioned/` |
| `bed.state_changed` | `bronze/eventstream/bed.state_changed/` |
| `bed.assigned` | `bronze/eventstream/bed.assigned/` |
| `forecast.published` | `bronze/eventstream/forecast.published/` |
| `discharge.scored` | `bronze/eventstream/discharge.scored/` |
| `discharge.recommended` | `bronze/eventstream/discharge.recommended/` |

## Execution mode

This notebook supports two execution modes, controlled by the `use_streaming`
parameter:

- **Batch (default, `use_streaming=False`)** — reads the Eventstream-materialised
  Delta table as a bounded snapshot. Idempotent when re-run against the same
  Eventstream retention window. Simpler for demo runs and CI replay.
- **Streaming (`use_streaming=True`)** — uses `spark.readStream.format('deltaLive')`
  bound to the Eventstream table (per Fabric's Delta Live streaming source
  pattern) with a per-batch dispatcher (`foreachBatch`) that appends to the
  per-`eventKind` bronze folders. `checkpoint_dir` must be set.

Both modes append (never overwrite) so bronze grows monotonically per envelope.

In [ ]:
# Parameters (Fabric injects overrides via the papermill-compatible 'parameters' tag).
eventstream_binding = 'bronze_eventstream_raw'          # lakehouse-managed Delta table populated by the Eventstream destination bound in T2.2
target_lakehouse = 'lh_ihzhhpf_sit'
bronze_root = 'Tables/bronze/eventstream'                # lakehouse-relative bronze root (one folder per eventKind)
run_id = 'run-manual-local'                              # overridden by pipeline / notebookutils
checkpoint_dir = 'Files/_checkpoints/bronze_eventstream' # only used when use_streaming=True
use_streaming = False                                    # False = batch snapshot; True = structured streaming via foreachBatch dispatcher

In [ ]:
# Registry of accepted event kinds (design spec §4.3). Envelopes carrying any other
# eventKind value are routed to `_unknown/` and surface as a warning; silver decides
# whether to reject or backfill.
EVENT_KINDS = [
    'encounter.admitted',
    'encounter.transitioned',
    'bed.state_changed',
    'bed.assigned',
    'forecast.published',
    'discharge.scored',
    'discharge.recommended',
]
UNKNOWN_BUCKET = '_unknown'

In [ ]:
from datetime import datetime, timezone
from pyspark.sql import DataFrame, functions as F

def _ingest_ts() -> str:
    return datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

def _route_and_write(df: DataFrame, ingest_ts: str) -> dict:
    """Route a bounded DataFrame of envelopes by `eventKind` and append each subset to bronze.

    Envelope columns are preserved verbatim. A `_lineage_ref` column stamps
    `<eventstream_binding>:<ingest_ts>` for silver-side traceability. The full
    envelope (`eventKind`, `eventId`, `hospitalId`, `simulatedAt`, `emittedAt`,
    `simRunId`, `seed`, `payload`) is retained — payload stays as a struct/string
    depending on how the Eventstream materialisation typed it. Returns a per-kind
    row-count map for the summary cell.
    """
    if 'eventKind' not in df.columns:
        raise ValueError("Bronze source is missing the 'eventKind' routing column")
    stamped = df.withColumn('_lineage_ref', F.lit(f'{eventstream_binding}:{ingest_ts}'))
    # Batch mode overwrites per-kind partitions so a git rebuild is idempotent;
    # streaming appends because each micro-batch is a new increment.
    bronze_write_mode = 'append' if use_streaming else 'overwrite'
    # Batch overwrite must REPLACE the schema (a prior live Eventstream run may
    # have typed `payload` as a STRUCT; the git seed types it as a JSON string).
    # mergeSchema cannot reconcile struct-vs-string, so use overwriteSchema in
    # batch and mergeSchema only for streaming appends.
    _schema_opt = ('mergeSchema', 'true') if use_streaming else ('overwriteSchema', 'true')
    counts: dict = {}
    # Known kinds
    for kind in EVENT_KINDS:
        subset = stamped.filter(F.col('eventKind') == F.lit(kind))
        n = subset.count()
        counts[kind] = n
        if n == 0:
            continue
        tgt = f'{bronze_root}/{kind}'
        (subset.write
               .format('delta')
               .mode(bronze_write_mode)
               .option(*_schema_opt)
               .save(tgt))
    # Unknown-kind bucket — kept so we do not silently drop malformed envelopes.
    unknown = stamped.filter(~F.col('eventKind').isin(EVENT_KINDS))
    n_unknown = unknown.count()
    counts[UNKNOWN_BUCKET] = n_unknown
    if n_unknown > 0:
        print(f'WARN bronze: {n_unknown} envelope(s) with unknown eventKind routed to {bronze_root}/{UNKNOWN_BUCKET}/')
        (unknown.write
                .format('delta')
                .mode(bronze_write_mode)
                .option(*_schema_opt)
                .save(f'{bronze_root}/{UNKNOWN_BUCKET}'))
    return counts

In [ ]:
# Main ingest — batch or streaming.
ingest_ts = _ingest_ts()

if use_streaming:
    # Fabric Delta Live streaming source pattern — the Eventstream destination exposes
    # a Delta Live table; readStream binds to it and foreachBatch dispatches per micro-batch.
    stream_df = (spark.readStream
                     .format('deltaLive')
                     .option('table', eventstream_binding)
                     .load())

    def _dispatch(batch_df, batch_id):
        batch_counts = _route_and_write(batch_df, _ingest_ts())
        print(f'stream batch_id={batch_id} routed: {batch_counts}')

    query = (stream_df.writeStream
                       .foreachBatch(_dispatch)
                       .option('checkpointLocation', checkpoint_dir)
                       .trigger(availableNow=True)  # drain then stop — pipeline schedules the next run
                       .start())
    query.awaitTermination()
    results = {'mode': 'streaming', 'note': 'per-batch counts printed above; trigger=availableNow'}
else:
    # Batch snapshot fallback — idempotent replay of whatever the Eventstream has
    # materialised so far. Bronze targets are overwritten per-kind in batch mode,
    # so re-running the notebook against the same source rebuilds cleanly without
    # duplicating rows (git-reproducible from the committed synthetic seed).
    src_df = spark.read.format('delta').load(f'Tables/{eventstream_binding}') \
                if not eventstream_binding.startswith('Tables/') \
                else spark.read.format('delta').load(eventstream_binding)
    counts = _route_and_write(src_df, ingest_ts)
    results = {'mode': 'batch', 'ingest_ts': ingest_ts, 'per_kind_counts': counts}

In [ ]:
# Summary.
print(f'Bronze eventstream ingestion summary (run_id={run_id})')
print('-' * 72)
print(f'mode = {results.get("mode")}')
if results.get('mode') == 'batch':
    print(f'ingest_ts = {results["ingest_ts"]}')
    for kind in EVENT_KINDS + [UNKNOWN_BUCKET]:
        print(f'  {kind:<32s} rows={results["per_kind_counts"].get(kind, 0)}')
else:
    print(results.get('note'))

## Routing self-test (in-memory)

Simulate a small batch of envelopes covering 3 sample event kinds plus one
malformed kind, and verify `_route_and_write`'s counting/routing logic without
touching the Eventstream source or writing to Delta. Uses an in-memory monkey-patch
of the write path so no lakehouse artefacts are produced.

**Expected output**: per-kind counts `{encounter.admitted: 2, bed.state_changed: 1,
discharge.scored: 1, _unknown: 1}` and `ROUTING TEST: PASSED`.

In [ ]:
from pyspark.sql import Row

test_rows = [
    Row(eventKind='encounter.admitted',  eventId='e1', hospitalId='H_USZ',  simRunId='sr1', seed=42, simulatedAt='2027-01-15T10:00:00Z', emittedAt='2026-07-03T00:00:00Z', payload='{}'),
    Row(eventKind='encounter.admitted',  eventId='e2', hospitalId='H_LUKS', simRunId='sr1', seed=42, simulatedAt='2027-01-15T10:05:00Z', emittedAt='2026-07-03T00:00:00Z', payload='{}'),
    Row(eventKind='bed.state_changed',   eventId='e3', hospitalId='H_USZ',  simRunId='sr1', seed=42, simulatedAt='2027-01-15T10:06:00Z', emittedAt='2026-07-03T00:00:00Z', payload='{}'),
    Row(eventKind='discharge.scored',    eventId='e4', hospitalId='H_SZB',  simRunId='sr1', seed=42, simulatedAt='2027-01-15T10:07:00Z', emittedAt='2026-07-03T00:00:00Z', payload='{}'),
    Row(eventKind='garbage.event',       eventId='e5', hospitalId='H_USZ',  simRunId='sr1', seed=42, simulatedAt='2027-01-15T10:08:00Z', emittedAt='2026-07-03T00:00:00Z', payload='{}'),
]
test_df = spark.createDataFrame(test_rows)

# Monkey-patch the writer for the test only — count subsets without hitting Delta.
captured = {}
def _fake_route_and_write(df, ingest_ts):
    stamped = df.withColumn('_lineage_ref', F.lit(f'test:{ingest_ts}'))
    counts = {}
    for kind in EVENT_KINDS:
        counts[kind] = stamped.filter(F.col('eventKind') == F.lit(kind)).count()
    counts[UNKNOWN_BUCKET] = stamped.filter(~F.col('eventKind').isin(EVENT_KINDS)).count()
    return counts

captured = _fake_route_and_write(test_df, _ingest_ts())

assert captured['encounter.admitted'] == 2, f"encounter.admitted expected 2, got {captured['encounter.admitted']}"
assert captured['bed.state_changed'] == 1, f"bed.state_changed expected 1, got {captured['bed.state_changed']}"
assert captured['discharge.scored'] == 1, f"discharge.scored expected 1, got {captured['discharge.scored']}"
assert captured[UNKNOWN_BUCKET] == 1, f"_unknown expected 1, got {captured[UNKNOWN_BUCKET]}"
# Ensure no rows leaked into unused kinds.
for kind in ('encounter.transitioned', 'bed.assigned', 'forecast.published', 'discharge.recommended'):
    assert captured[kind] == 0, f'{kind} expected 0, got {captured[kind]}'

print('ROUTING TEST: PASSED')
print(f'  captured counts: {captured}')